In [79]:
!pip install torch transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [80]:
import json
import os
import torch
CLASSES = {
    "Adware": 0,
    "Backdoor": 1,
    "Botnet": 2,
    "CGI": 3,
    "Code-execution": 4,
    "DDos": 5,
    "Dir-Traversal": 6,
    "Dos": 7,
    "Info-Disclosure": 8,
    "Injection": 9,
    "Other": 10,
    "Overflow": 11,
    "Ransomware": 12,
    "Remote-file-Inclusion": 13,
    "Scanner": 14,
    "Spyware": 15,
    "Trojan": 16,
    "Virus": 17,
    "Webshell": 18,
    "Worm": 19,
    "XSS": 20
}
INV_CLASSES = {v: k for k, v in CLASSES.items()}
CONCEPTS= ["ip", "injection"]
CLASSES_TO_EXAMINE = ["Adware", "Scanner", "Spyware", "Trojan", "XSS", "Remote-file-Inclusion", "Overflow", "Injection", "Info-Disclosure", "Dir-Traversal", "Code-execution", "CGI", "Ransomware", "Botnet", "Backdoor"]
MODEL_NAME = "./codebert-base-mlm"


from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [3]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [81]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data
import numpy as np
import random

In [ ]:
# Estrazione delle feature
f = open("./data/packet_inspection/packets_dataset.jsonl", "r")
ai_dataset = [json.loads(line) for line in f.readlines()]
f.close()

f = open("./data/packet_inspection/anomalous_packets.jsonl", "r")
val_test_dataset = [json.loads(line) for line in f.readlines()]
f.close()

train_dataset = random.sample(val_test_dataset, 1000)

val_dataset = random.sample([v for v in val_test_dataset if v not in train_dataset], 100)
test_dataset = ai_dataset

tokenized_inputs = tokenizer([s["text"] for s in train_dataset], padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**tokenized_inputs)
    hidden_states = outputs.hidden_states
    # Prendi l'ultimo layer nascosto
    activations = [h.numpy() for h in hidden_states]  # Converti in numpy array
print("Done training inputs")

val_tokenized_inputs = tokenizer([s["text"] for s in val_dataset], padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    val_outputs = model(**val_tokenized_inputs)
    val_hidden_states = val_outputs.hidden_states
    # Prendi l'ultimo layer nascosto
    val_activations = [h.numpy() for h in val_hidden_states]
print("Done validating inputs")

test_tokenized_inputs = tokenizer([s["text"] for s in test_dataset], padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    test_outputs = model(**test_tokenized_inputs)
    test_hidden_states = test_outputs.hidden_states
    # Prendi l'ultimo layer nascosto
    test_activations = [h.numpy() for h in test_hidden_states]
print("Done testing inputs")

Done training inputs
Done validating inputs
Done testing inputs


In [83]:
class SAE(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(SAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

In [ ]:
def calculate_sae_loss(x, W_enc, b_enc, W_dec, b_dec, lambd):
    """
    Calcola la loss function per uno Sparse AutoEncoder come specificato nell'immagine.

    Args:
        x (torch.Tensor): Tensore dei dati di input, di forma (batch_size, D).
        W_enc (torch.Tensor): Pesi dell'encoder, di forma (F, D).
        b_enc (torch.Tensor): Bias dell'encoder, di forma (F,).
        W_dec (torch.Tensor): Pesi del decoder, di forma (D, F).
        b_dec (torch.Tensor): Bias del decoder, di forma (D,).
        lambd (float): Parametro di regolarizzazione.

    Returns:
        torch.Tensor: Il valore scalare della loss.
    """
    # 1. Calcolo delle attivazioni dei feature (f_i(x) = ReLU(W_enc * x + b_enc))
    # Il prodotto matriciale (matmul) deve tenere conto delle dimensioni
    # Il documento indica W_enc come (F, D), quindi lo usiamo direttamente
    # per matmul(W_enc, x.T) o usiamo x con matmul(x, W_enc.T)
    # Calcola le attivazioni delle feature: per ogni feature i, W_enc[i] @ x
    # x: (batch_size, 768), W_enc: (6144, 768)
    # Risultato: (batch_size, 6144)
    feature_activations = torch.relu(torch.matmul(x, W_enc.T) + b_enc)
    # print(feature_activations.shape)

    # 2. Ricostruzione dell'input (x_hat = W_dec * f(x) + b_dec)
    # W_dec è di forma (D, F) e feature_activations di forma (batch_size, F)
    x_hat = torch.matmul(feature_activations, W_dec.T) + b_dec
    
    # 3. Calcolo del termine di ricostruzione (L2 norm)
    # ||x - x_hat||_2^2
    reconstruction_loss = torch.mean(torch.sum((x - x_hat)**2, dim=1))
    
    # 4. Calcolo del termine di penalizzazione (L1 norm)
    # lambda * sum(f_i(x) * ||W_dec_i||_2)
    # La tua immagine mostra un L2 norm sui pesi del decoder,
    # ma una regolarizzazione L1 sulle attivazioni.
    # Spieghiamola così: la somma dei valori assoluti delle attivazioni moltiplicata per il peso
    # e una lambda.
    # W_dec è (D,F), quindi W_dec_i (cioè W_dec[:,i]) è un vettore colonna D-dimensionale.
    # La norma ||W_dec_i||_2 è la norma L2 di questa colonna.
    # Lo sum sulle colonne ci dà un vettore (F,) con la norma L2 di ogni colonna di W_dec.
    W_dec_norms = torch.norm(W_dec, p=2, dim=0)
    
    # Calcola la penalizzazione: somma del prodotto delle attivazioni e delle norme
    l1_penalty = torch.mean(torch.matmul(feature_activations, W_dec_norms))

    # 5. Calcolo della loss totale
    total_loss = reconstruction_loss + lambd * l1_penalty

    return total_loss


def get_feature_directions(W_dec):
    """
    Calcola i vettori di direzione delle feature a partire dalla matrice dei pesi del decoder.

    Args:
        W_dec (torch.Tensor): Pesi del decoder di forma (D, F), dove
                              D è la dimensione residua e F la dimensione delle feature.

    Returns:
        torch.Tensor: I vettori delle feature (direzioni normalizzate), di forma (D, F).
    """
    # Calcola la norma L2 di ogni colonna (dim=0) della matrice W_dec.
    # Aggiunge 1e-8 per evitare divisioni per zero.
    norms = torch.linalg.norm(W_dec, dim=0)
    
    # Normalizza ogni colonna (vettore di feature) dividendo per la sua norma.
    feature_directions = W_dec / (norms + 1e-8)
    
    return feature_directions

def get_feature_activations(x, W_enc, b_enc, W_dec):
    """
    Calcola le attivazioni delle feature per un dato input x.

    Args:
        x (torch.Tensor): Tensore dei dati di input, di forma (batch_size, D).
        W_enc (torch.Tensor): Pesi dell'encoder, di forma (F, D).
        b_enc (torch.Tensor): Bias dell'encoder, di forma (F,).
        W_dec (torch.Tensor): Pesi del decoder, di forma (D, F).

    Returns:
        torch.Tensor: Le attivazioni delle feature, di forma (batch_size, F).
    """
    # Calcolo delle attivazioni dei feature (f_i(x) = ReLU(W_enc * x + b_enc))
    f_x = torch.relu(torch.matmul(x, W_enc.T) + b_enc)
    
    norm_W_dec = torch.linalg.norm(W_dec, dim=0)
    
    feature_activations = f_x * norm_W_dec
    
    return feature_activations

def init_decoder_weights(input_dim, hidden_dim, l2_norm=0.1):
    # Crea una matrice random di shape (input_dim, hidden_dim)
    W_d = torch.randn(input_dim, hidden_dim)
    # Normalizza ogni colonna a norma L2 = l2_norm
    W_d = W_d / W_d.norm(dim=0, keepdim=True) * l2_norm
    return W_d

In [ ]:
from itertools import product
from tqdm import tqdm

SKIP = True
os.makedirs("saved_models", exist_ok=True)

input_dim = 768  # Dimension of RoBERTa embeddings
layer = 10       # Fissa il layer da usare
beta = 5.0       # Peso della penalizzazione
lr = 5e-5     # Fissa il learning rate
hidden_dims = [768, 768*2, 768*4, 768*8, 768*16, 768*32]  # Diverse dimensioni da provare

if not SKIP:
    previous_results = []
    try:
        with open("./saved_models/training_results.json", "r") as f:
            for l in f.readlines():
                previous_results.append(json.loads(l))
    except FileNotFoundError:
        pass
    results = []

    for hidden_dim in hidden_dims:
        # Prepara DataLoader
        X_train = activations[layer]
        train_dataset = data.TensorDataset(torch.from_numpy(X_train))
        train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)
        
        sae = SAE(input_dim, hidden_dim)
        
        # Inizializza i pesi del sae
        with torch.no_grad():
            # Inizializza decoder
            W_d = init_decoder_weights(input_dim, hidden_dim)
            sae.decoder[0].weight.copy_(W_d)  # PyTorch usa shape (out, in)
            # Inizializza encoder come trasposta
            sae.encoder[0].weight.copy_(W_d.t())
        optimizer = optim.Adam(sae.parameters(), lr=lr)
        epochs = 200

        # Early stopping parameters
        early_stop_patience = 5
        best_val_loss = float('inf')
        epochs_no_improve = 0
        avg_loss = float('inf')
        # Early stopping training loop
        for epoch in tqdm(range(epochs), desc=f"Training SAE Layer {layer} Hidden {hidden_dim}", postfix={"loss": avg_loss}):
            # Imposta beta: parte basso, cresce linearmente fino a beta target dopo il 10% delle epoche
            if epoch < int(epochs * 0.1):
                curr_beta = beta * (epoch / (epochs * 0.1))
            else:
                curr_beta = beta

            epoch_loss = 0
            sae.train()
            for batch_idx, (data_batch,) in enumerate(train_loader):
                reconstructed_data, encoded_activations = sae(data_batch)
                encoder_weights = sae.encoder[0].weight
                decoder_weights = sae.decoder[0].weight
                loss = calculate_sae_loss(
                    data_batch,
                    reconstructed_data,
                    encoded_activations,
                    decoder_weights,
                    lambd=curr_beta
                )
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / len(X_train)

            # Validazione
            val_X = val_activations[layer]
            val_dataset = data.TensorDataset(torch.from_numpy(val_X))
            val_loader = data.DataLoader(val_dataset, batch_size=32, shuffle=False)

            # Validation loss for early stopping
            sae.eval()
            val_loss = 0
            with torch.no_grad():
                for val_batch, in val_loader:
                    reconstructed_data, encoded_activations = sae(val_batch)
                    encoder_weights = sae.encoder[0].weight
                    decoder_weights = sae.decoder[0].weight
                    
                    v_loss = calculate_sae_loss(
                    val_batch,
                    reconstructed_data,
                    encoded_activations,
                    decoder_weights,
                    lambd=beta
                )
                    val_loss += v_loss.item()
                avg_val_loss = val_loss / len(val_X)
                

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                # Optionally save best model weights here
                best_model_state = sae.state_dict()
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= early_stop_patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    # Restore best model weights
                    sae.load_state_dict(best_model_state)
                    break
            
        print(f"Final training loss: {avg_loss:.4f}, Best validation loss: {best_val_loss:.4f}")

        # Test
        test_X = test_activations[layer]
        test_dataset = data.TensorDataset(torch.from_numpy(test_X))
        test_loader = data.DataLoader(test_dataset, batch_size=32, shuffle=False)

        test_loss = 0
        with torch.no_grad():
            for test_batch, in test_loader:
                reconstructed_data, encoded_activations = sae(test_batch)
                encoder_weights = sae.encoder[0].weight
                decoder_weights = sae.decoder[0].weight
                
                t_loss = calculate_sae_loss(
                    test_batch,
                    reconstructed_data,
                    encoded_activations,
                    decoder_weights,
                    lambd=beta
                )
                test_loss += t_loss.item()
            avg_test_loss = test_loss / len(test_X)
        print(f"Test loss: {avg_test_loss:.4f}")
        
        # Salva ogni risultato
        x = {"layer": layer, "hidden_dim": hidden_dim, "beta": beta, "lr": lr, "avg_loss": avg_loss}
        with open("saved_models/training_results.json", "a") as f:
            json.dump(x, f)
            f.write("\n")
        results.append(x)

        # Salva il modello per ogni combinazione
        torch.save(sae.state_dict(), f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}.pt")

    print("Tutti i modelli e dettagli di training salvati in 'saved_models/' e 'training_results.json'")


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
validations = random.sample([d for d in val_test_dataset if d not in train_dataset], 10)
inputs = [s["text"] for s in validations]
tokens_id = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt")
activations =  model(**tokens_id).hidden_states

num_samples, num_tokens, _ = activations[layer].shape
tokens_str = [tokenizer.tokenize(s, truncation=True,padding="max_length", max_length=num_tokens) for s in inputs]

print("Done gathering inputs")


In [ ]:
import umap

# Analyzing feature directions for all saes

hidden_dims = [768, 768*2, 768*4, 768*8, 768*16, 768*32] 
layer = 10
beta = 5.0
lr = 5e-5

feature_directions_dict = {}

for hidden_dim in hidden_dims:
    sae = SAE(input_dim, hidden_dim)
    sae.load_state_dict(torch.load(f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}.pt"))
    encoder_weights = sae.encoder[0].weight
    decoder_weights = sae.decoder[0].weight
    feature_directions = get_feature_directions(decoder_weights)
    feature_directions_dict[hidden_dim] = feature_directions

    test_X_tensor = activations[layer]
    with torch.no_grad():
        sae.eval()
        _ , feature_activations = sae(test_X_tensor)

    nonzero_counts = (feature_activations != 0).sum(dim=2)  # shape: (100, 512)
    flattened_nonzero_counts = nonzero_counts.flatten()
    sparsity = flattened_nonzero_counts.float().mean().item()
    print(f"[Dim {hidden_dim}] Sparsità media (numero di feature attive per singolo token): {sparsity:.4f}")

    mean_activations = feature_activations.mean(dim=(0,1)) # shape: (hidden_dim)
    live_features = (mean_activations!=0).float().mean().item()
    print(f"[Dim {hidden_dim}] Valore medio feature attive: {live_features:.4f}")

    # Calcola per ogni feature (sull'ultima dimensione) se è sempre zero su tutti i token di tutti i sample
    # feature_activations: (num_samples, num_tokens, hidden_dim)
    # dead_features_mask: (hidden_dim,) True se la feature è sempre zero
    print(feature_activations)
    dead_features = (feature_activations == 0).all(dim=(0, 1)).sum().item()
    print(f"[Dim {hidden_dim}] Numero di dead features (mai attivate): {dead_features} su {len(mean_activations)}")

    top_features = torch.topk(mean_activations, k=10)
    print(f"[Dim {hidden_dim}] Top 10 feature (concetti) più attive:")
    for idx, value in zip(top_features.indices.tolist(), top_features.values.tolist()):
        print(idx, value)
        print(f"Feature {idx}: attivazione media = {value:.4f}")
        

import matplotlib.pyplot as plt

reducer = umap.UMAP(n_components=2, random_state=42)
colors = ['red', 'orange', 'green', 'blue', 'purple', 'black']
plt.figure(figsize=(10, 8))

min_hd = min(hidden_dims)
base_size = 20
# scala la dimensione dei punti in modo decrescente al crescere di hidden_dim
sizes = [max(5, base_size * (min_hd / hd) ** 0.5) for hd in hidden_dims]

for i, hidden_dim in enumerate(hidden_dims):
    fd = feature_directions_dict[hidden_dim].detach().cpu().numpy().T  # shape: (hidden_dim, input_dim)
    embedding = reducer.fit_transform(fd)
    plt.scatter(
        embedding[:, 0], embedding[:, 1],
        label=f"hidden_dim={hidden_dim}",
        alpha=0.6, s=sizes[i], color=colors[i % len(colors)]
    )

plt.title("UMAP 2D delle feature_directions per diversi hidden_dim")
plt.legend(title="Hidden dim")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.show()

In [ ]:
hidden_dim = 768*8
sae = SAE(input_dim, hidden_dim)
sae.load_state_dict(torch.load(f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}.pt"))
encoder_weights = sae.encoder[0].weight
decoder_weights = sae.decoder[0].weight
feature_directions = get_feature_directions(decoder_weights)
_, feature_activations = sae.eval()(torch.from_numpy(activations[layer]))

token_to_feature_activations = {}
num_samples, num_tokens, _ = activations[layer].shape
# print(activations[layer].shape)
# print(len(tokens_str[0]))
for sample_idx in range(num_samples):
    for token_idx in range(num_tokens):
        if token_idx == 0:
            token_to_feature_activations[(sample_idx, "CLS", token_idx)] = feature_activations[sample_idx, token_idx].detach().cpu().numpy()
        else:
            if tokens_id.attention_mask[sample_idx][token_idx] == 1:
                if tokens_str[sample_idx][token_idx-1] == "<pad>":
                    token_to_feature_activations[(sample_idx, "</s>", token_idx)] = feature_activations[sample_idx, token_idx].detach().cpu().numpy()
                else:
                    token_to_feature_activations[(sample_idx, tokens_str[sample_idx][token_idx-1], token_idx)] = feature_activations[sample_idx, token_idx].detach().cpu().numpy()
            else:
                continue

# print(token_to_feature_activations.keys())
first_key = next(iter(token_to_feature_activations))
print(len(token_to_feature_activations[first_key]))

In [ ]:
from collections import defaultdict
from tqdm import tqdm
# Crea un dizionario che, per ogni feature, raccoglie i token stringa che hanno attivazione non zero per quella feature
# Ora includi anche il token_idx nella tupla

feature_to_tokens = defaultdict(list)  # feature_idx -> lista di tuple (token stringa, sample_idx, token_idx, activation_value)

for (sample_idx, token_str, token_idx), activations in tqdm(token_to_feature_activations.items()):
    for feature_idx, activation_value in enumerate(activations):
        if activation_value != 0:
            feature_to_tokens[feature_idx].append((token_str, sample_idx, token_idx, activation_value))

# Ordina i token per ogni feature in base al valore di attivazione (decrescente)
for feature_idx in feature_to_tokens:
    feature_to_tokens[feature_idx] = sorted(
        feature_to_tokens[feature_idx],
        key=lambda x: abs(x[3]),  # ordina per valore assoluto dell'attivazione
        reverse=True
    )


# Ora feature_to_tokens[feature_idx] contiene l'insieme dei token stringa attivati per ogni feature
# Esempio: mostra i primi 5 feature e i loro token associati
for feature_idx in list(feature_to_tokens.keys())[:10]:
    print(f"Feature {feature_idx}: {list(feature_to_tokens[feature_idx])[:10]}")

In [ ]:
for feature_idx in list(feature_to_tokens.keys())[:10]:
    print(f"Feature {feature_idx}: {list(feature_to_tokens[feature_idx])[:50]}")

In [ ]:
import json

# Funzione per serializzare i dati in formato richiesto
def serialize_feature_to_tokens(feature_to_tokens):
    result = {}
    for feature_idx, tokens in feature_to_tokens.items():
        result[str(feature_idx)] = [
            {
                "str": token_str,
                "sample_id": int(sample_idx),
                "token_id": int(token_idx),
                "activation": float(activation_value)
            }
            for token_str, sample_idx, token_idx, activation_value in tokens
        ]
    return result

serialized = serialize_feature_to_tokens(feature_to_tokens)

filename = f"./sae_results/feature_to_tokens_hidden{hidden_dim}_layer{layer}_beta{beta}_lr{lr}.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(serialized, f, ensure_ascii=False, indent=2)
print(f"feature_to_tokens salvato in {filename}")

In [ ]:
import json

output = []
num_samples, num_tokens, hidden_dim = feature_activations.shape

for sample_idx in tqdm(range(num_samples)):
    tokens_json = []
    # Primo token: CLS
    nonzero = feature_activations[sample_idx, 0].nonzero().squeeze().tolist()
    if isinstance(nonzero, int):
        nonzero = [nonzero]
    activations = [
        (int(i), float(feature_activations[sample_idx, 0, i].item()))
        for i in nonzero
        if feature_activations[sample_idx, 0, i] != 0
    ]
    tokens_json.append({
        "token_idx": 0,
        "token_str": "CLS",
        "activations": activations
    })
    # print({
    #     "token_idx": 0,
    #     "token_str": "CLS",
    #     "activations": activations
    # })
    # Token successivi
    for token_idx, token in enumerate(tokens_str[sample_idx]):
        fa_idx = token_idx + 1
        if fa_idx >= num_tokens:
            break
        if token == "<pad>":
            nonzero = feature_activations[sample_idx, fa_idx].nonzero().squeeze().tolist()
            if isinstance(nonzero, int):
                nonzero = [nonzero]
            activations = [
                (int(i), float(feature_activations[sample_idx, fa_idx, i].item()))
                for i in nonzero
                if feature_activations[sample_idx, fa_idx, i] != 0
            ]
            tokens_json.append({
                "token_idx": fa_idx,
                "token_str": "</s>",
                "activations": activations
            })
            break  # Stop at first <pad>
        else:
            nonzero = feature_activations[sample_idx, fa_idx].nonzero().squeeze().tolist()
            if isinstance(nonzero, int):
                nonzero = [nonzero]
            activations = [
                (int(i), float(feature_activations[sample_idx, fa_idx, i].item()))
                for i in nonzero
                if feature_activations[sample_idx, fa_idx, i] != 0
            ]
            tokens_json.append({
                "token_idx": fa_idx,
                "token_str": token,
                "activations": activations
            })
    output.append({
        "sample_index": sample_idx,
        "tokens": tokens_json
    })

# print(json.dumps(output[0], ensure_ascii=False, indent=2))

with open("tokens_with_activations_sparse.json", "w") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Output JSON creato con {len(output)} samples.")